In [1]:
import numpy as np
import matplotlib.pyplot as plt

from scipy.stats import norm
from scipy.spatial import cKDTree

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    ConstantKernel,
    Matern,
    WhiteKernel
)

# --------------------------------------------------
# Function 2 - Week 7
# --------------------------------------------------
# The Week 7 .npy files already contain the original
# Function 2 data plus Weeks 1-6 exactly once.
#
# Strategy:
# - fit GP to all accumulated observations
# - inspect ARD lengthscales
# - derive candidate widths from GP
# - generate local + wider + global candidates
# - calibrate EI and UCB before choosing the query

In [2]:
X = np.load("function2/initial_inputs.npy")
Y = np.load("function2/initial_outputs.npy").reshape(-1)

print("X shape:", X.shape)
print("Y shape:", Y.shape)

assert len(X) == len(Y)
assert X.shape[1] == 2

best_idx = np.argmax(Y)
best_x = X[best_idx]
best_y = Y[best_idx]

print("\nCurrent best observed input:", best_x)
print("Current best observed output:", best_y)

X shape: (16, 2)
Y shape: (16,)

Current best observed input: [0.711177 1.      ]
Current best observed output: 0.679663


In [3]:
X = np.load("function2/initial_inputs.npy")
Y = np.load("function2/initial_outputs.npy").reshape(-1)

print("X shape:", X.shape)
print("Y shape:", Y.shape)

assert len(X) == len(Y)
assert X.shape[1] == 2

best_idx = np.argmax(Y)
best_x = X[best_idx]
best_y = Y[best_idx]

print("\nCurrent best observed input:", best_x)
print("Current best observed output:", best_y)

X shape: (16, 2)
Y shape: (16,)

Current best observed input: [0.711177 1.      ]
Current best observed output: 0.679663


In [4]:
kernel = (
    ConstantKernel(1.0, (1e-3, 1e3))
    * Matern(
        length_scale=np.ones(2) * 0.2,
        length_scale_bounds=(0.01, 2.0),
        nu=2.5
    )
    + WhiteKernel(
        noise_level=1e-5,
        noise_level_bounds=(1e-8, 1e-1)
    )
)

gp = GaussianProcessRegressor(
    kernel=kernel,
    normalize_y=True,
    n_restarts_optimizer=20,
    random_state=42
)

gp.fit(X, Y)

print("\nFitted kernel:")
print(gp.kernel_)


Fitted kernel:
0.887**2 * Matern(length_scale=[0.0256, 1.27], nu=2.5) + WhiteKernel(noise_level=1e-08)


/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:440: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-08. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


In [5]:
lengthscales = gp.kernel_.k1.k2.length_scale

inverse_ls = 1 / lengthscales
sensitivity = inverse_ls / inverse_ls.sum()

print("\nARD lengthscales:", lengthscales)
print("Normalised inverse-lengthscale sensitivity:", sensitivity)


ARD lengthscales: [0.02556369 1.26883083]
Normalised inverse-lengthscale sensitivity: [0.98025046 0.01974954]


In [6]:
local_scale = np.clip(
    0.25 * lengthscales,
    0.02,
    0.10
)

wide_scale = np.clip(
    0.50 * lengthscales,
    0.05,
    0.20
)

print("\nLocal search widths:", local_scale)
print("Wider search widths:", wide_scale)


Local search widths: [0.02 0.1 ]
Wider search widths: [0.05 0.2 ]


In [7]:
rng = np.random.default_rng(42)

local_candidates = best_x + rng.normal(
    0,
    local_scale,
    size=(40000, 2)
)

wide_candidates = best_x + rng.normal(
    0,
    wide_scale,
    size=(20000, 2)
)

global_candidates = rng.uniform(
    0,
    1,
    size=(10000, 2)
)

candidates = np.vstack([
    local_candidates,
    wide_candidates,
    global_candidates
])

candidates = np.clip(candidates, 0, 1)

print("Generated candidates:", len(candidates))

Generated candidates: 70000


In [8]:
tree = cKDTree(X)

distance, _ = tree.query(candidates, k=1)

candidates = candidates[distance > 0.01]

print("Candidates after filtering:", len(candidates))

Candidates after filtering: 56338


In [9]:
mu, sigma = gp.predict(
    candidates,
    return_std=True
)

In [10]:
def expected_improvement(mu, sigma, best_y, xi=0.0):

    improvement = mu - best_y - xi

    Z = np.zeros_like(mu)

    valid = sigma > 1e-12
    Z[valid] = improvement[valid] / sigma[valid]

    EI = np.zeros_like(mu)

    EI[valid] = (
        improvement[valid] * norm.cdf(Z[valid])
        + sigma[valid] * norm.pdf(Z[valid])
    )

    return EI

In [11]:
print("\nEI calibration:\n")

for xi in [0.0, 0.001, 0.005, 0.01, 0.02]:

    EI_test = expected_improvement(
        mu,
        sigma,
        best_y,
        xi
    )

    idx = np.argmax(EI_test)

    print(
        f"xi={xi}",
        "\n candidate =", candidates[idx],
        "\n mean =", round(mu[idx], 6),
        "\n std =", round(sigma[idx], 6),
        "\n EI =", round(EI_test[idx], 8),
        "\n"
    )


EI calibration:

xi=0.0 
 candidate = [0.6882722  0.01369931] 
 mean = 0.508377 
 std = 0.137945 
 EI = 0.0071 

xi=0.001 
 candidate = [0.6882722  0.01369931] 
 mean = 0.508377 
 std = 0.137945 
 EI = 0.00699349 

xi=0.005 
 candidate = [0.6882722  0.01369931] 
 mean = 0.508377 
 std = 0.137945 
 EI = 0.00658061 

xi=0.01 
 candidate = [0.6882722  0.01369931] 
 mean = 0.508377 
 std = 0.137945 
 EI = 0.00609317 

xi=0.02 
 candidate = [0.6882722  0.01369931] 
 mean = 0.508377 
 std = 0.137945 
 EI = 0.00520832 



In [12]:
print("\nUCB calibration:\n")

for beta in [0.1, 0.25, 0.5, 1.0, 1.5]:

    UCB = mu + beta * sigma
    idx = np.argmax(UCB)

    print(
        f"beta={beta}",
        "\n candidate =", candidates[idx],
        "\n mean =", round(mu[idx], 6),
        "\n std =", round(sigma[idx], 6),
        "\n UCB =", round(UCB[idx], 6),
        "\n"
    )


UCB calibration:

beta=0.1 
 candidate = [0.7113181  0.98999143] 
 mean = 0.678799 
 std = 0.001602 
 UCB = 0.67896 

beta=0.25 
 candidate = [0.7113181  0.98999143] 
 mean = 0.678799 
 std = 0.001602 
 UCB = 0.6792 

beta=0.5 
 candidate = [0.71166486 0.98974933] 
 mean = 0.678448 
 std = 0.00237 
 UCB = 0.679633 

beta=1.0 
 candidate = [0.71156948 0.91303056] 
 mean = 0.670554 
 std = 0.011909 
 UCB = 0.682463 

beta=1.5 
 candidate = [0.6882722  0.01369931] 
 mean = 0.508377 
 std = 0.137945 
 UCB = 0.715294 



In [13]:
mean_idx = np.argmax(mu)

print("\nHighest predicted mean:")
print("candidate =", candidates[mean_idx])
print("mean =", mu[mean_idx])
print("std =", sigma[mean_idx])


Highest predicted mean:
candidate = [0.7113181  0.98999143]
mean = 0.6787994159287263
std = 0.001602396948699824


In [14]:
# --------------------------------------------------
# Final Function 2 Week 7 selection
# --------------------------------------------------
#
# EI consistently selected a low-x2 candidate with a much lower predicted
# mean but very high uncertainty, indicating that the acquisition was mainly
# being driven by exploration.
#
# The fitted ARD Matern kernel gave lengthscales [0.0256, 1.27], suggesting
# that the GP models x1 as much more sensitive than x2.
#
# The highest predicted mean and low-exploration UCB (beta=0.1 and 0.25)
# all selected the same region. I therefore use UCB with beta=0.1, which
# retains an uncertainty component while giving more weight to predicted
# performance.

beta = 0.1

UCB = mu + beta * sigma
final_idx = np.argmax(UCB)

week7_candidate = candidates[final_idx]

print("Week 7 Function 2 candidate:")
print(week7_candidate)

print("\nPredicted mean:", mu[final_idx])
print("Predicted std:", sigma[final_idx])
print("UCB:", UCB[final_idx])

portal = "-".join(f"{x:.6f}" for x in week7_candidate)

print("\nPortal format:")
print(portal)

Week 7 Function 2 candidate:
[0.7113181  0.98999143]

Predicted mean: 0.6787994159287263
Predicted std: 0.001602396948699824
UCB: 0.6789596556235963

Portal format:
0.711318-0.989991
